# 09b — Esercitazione: libreria client OpenAI

**Corso**: Programmazione di Applicazioni Intelligenti  
**Lezione 09** — Dal Client OpenAI al Function Calling e MCP  
**Blocco 2** — Esercitazione (30 min)

In questa esercitazione metterai in pratica i concetti del Blocco 1:
- Configurare il client e fare la prima chiamata
- Costruire un chatbot multi-turno
- Usare lo structured output con Pydantic
- (Bonus) Esplorare i reasoning token

Ogni esercizio ha una **traccia con `# TODO`** da completare. Le soluzioni sono nel notebook `09c`.


---
## Setup

Esegui questa cella per configurare il client. È identica al Notebook 09a.


In [ ]:
# Setup (uguale per tutti gli esercizi)
!pip install -q openai

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
)

MODEL = "openai/gpt-oss-120b"
print(f"Client pronto! Modello: {MODEL}")


---
## Esercizio 1 — Setup e prima chiamata (5 min)

### Obiettivo
Fare una chiamata al modello con un **system prompt personalizzato** che modifica il comportamento dell'assistente, e ispezionare la risposta.

### Consegna
1. Crea un system prompt che dica al modello di comportarsi come un pirata (risposte in italiano, tono da pirata)
2. Fai una domanda qualsiasi (es. "Cos'è Python?")
3. Stampa la risposta
4. Stampa il conteggio dei token (prompt, completion, totali)


In [ ]:
# Esercizio 1 — Prima chiamata con system prompt personalizzato

# TODO: crea la chiamata con client.chat.completions.create()
# - model: usa la variabile MODEL
# - messages: lista con un messaggio system (pirata) e un messaggio user
response = # TODO

# TODO: stampa la risposta (response.choices[0].message.content)


# TODO: stampa i token consumati (prompt_tokens, completion_tokens, total_tokens)



---
## Esercizio 2 — Chatbot multi-turno (10 min)

### Obiettivo
Costruire un **loop interattivo** che simula una conversazione con memoria. Ad ogni turno il programma deve:
1. Leggere l'input dell'utente
2. Aggiungerlo alla cronologia
3. Chiamare l'API con tutta la cronologia
4. Stampare la risposta e il conteggio dei token

### Consegna
- Il chatbot parte con un system prompt a tua scelta
- Ad ogni turno, stampa il numero di **token totali** usati nella chiamata (per osservare la crescita)
- L'utente esce digitando `esci`
- **Bonus**: aggiungi un comando `/reset` che azzera la cronologia mantenendo il system prompt

### Suggerimento
La cronologia è una lista di dizionari `{"role": ..., "content": ...}`. Ad ogni turno aggiungi il messaggio user, fai la chiamata, e aggiungi la risposta assistant.


In [ ]:
# Esercizio 2 — Chatbot multi-turno

# System prompt iniziale
system_prompt = {"role": "system", "content": "Sei un assistente amichevole. Rispondi in italiano, in modo conciso."}

# TODO: inizializza la cronologia con il system prompt
cronologia = # TODO

print("Chatbot avviato! Scrivi 'esci' per uscire.")
print()

while True:
    # TODO: leggi l'input dell'utente con input()
    user_input = # TODO

    # Controlla se l'utente vuole uscire
    if user_input.lower() == "esci":
        print("Arrivederci!")
        break

    # BONUS: controlla se l'utente ha digitato /reset
    # TODO (opzionale): se user_input == "/reset", azzera la cronologia e continua

    # TODO: aggiungi il messaggio user alla cronologia
    # cronologia.append({"role": "user", "content": ...})

    # TODO: chiama l'API con la cronologia completa
    response = # TODO

    # TODO: estrai la risposta dall'oggetto response
    risposta = # TODO

    # TODO: aggiungi la risposta assistant alla cronologia
    # cronologia.append({"role": "assistant", "content": ...})

    # TODO: stampa la risposta
    # TODO: stampa il numero di token totali (response.usage.total_tokens)
    # Osserva come i token crescono ad ogni turno!


---
## Esercizio 3 — Output strutturato con Pydantic (10 min)

### Obiettivo
Usare lo **structured output** per estrarre informazioni strutturate dal modello e raccoglierle in una lista.

### Consegna
1. Definisci una classe Pydantic `CityInfo` con questi campi:
   - `name`: str — nome della città
   - `country`: str — paese
   - `population`: int — popolazione approssimativa
   - `famous_for`: List[str] — lista di cose per cui è famosa
   - `is_capital`: bool — se è una capitale
2. Chiama il modello con `response_format=CityInfo` per **3 città diverse** (es. Roma, Tokyo, New York)
3. Raccogli i risultati in una lista
4. Stampa una tabella riepilogativa

### Suggerimento
Usa `client.beta.chat.completions.parse()` con `response_format=CityInfo`.


In [ ]:
# Esercizio 3 — Structured output

from pydantic import BaseModel
from typing import List

# TODO: definisci la classe CityInfo con i campi richiesti
class CityInfo(BaseModel):
    pass  # TODO: sostituisci con i campi corretti


# Lista delle città da analizzare
citta = ["Roma", "Tokyo", "New York"]

# TODO: per ogni città, chiama il modello con structured output e raccogli i risultati
risultati = []

for nome_citta in citta:
    # TODO: chiama client.beta.chat.completions.parse()
    # - messages: chiedi al modello di fornire informazioni sulla città
    # - response_format: CityInfo
    response = # TODO

    # TODO: estrai l'oggetto parsed e aggiungilo a risultati
    info = # TODO
    risultati.append(info)

    print(f"Analizzata: {nome_citta}")


In [ ]:
# Stampa una tabella riepilogativa

# TODO: stampa i risultati in formato tabella
# Suggerimento: itera su risultati e stampa i campi allineati
print(f"{'Città':<15} {'Paese':<15} {'Popolazione':<15} {'Capitale':<10} {'Famosa per'}")
print("-" * 80)

for info in risultati:
    pass  # TODO: stampa una riga per ogni città


---
## Esercizio 4 (Bonus) — Reasoning visibile (5 min)

### Obiettivo
Confrontare la stessa domanda **con e senza reasoning** per osservare:
- Come cambia (o non cambia) la qualità della risposta
- Quanti token in più consuma il reasoning
- Se il contenuto del reasoning è visibile

### Consegna
1. Scegli una domanda che richiede ragionamento (es. un problema logico o matematico)
2. Fai **due chiamate**: una senza reasoning e una con `reasoning_effort="high"`
3. Confronta: risposta, token consumati, contenuto del reasoning


In [ ]:
# Esercizio 4 (Bonus) — Confronto con e senza reasoning

domanda = "Se ho 3 scatole e in ogni scatola ci sono 4 sacchetti, e in ogni sacchetto ci sono 5 biglie, quante biglie ho in totale?"

# --- Chiamata SENZA reasoning ---
# TODO: fai la chiamata standard (senza reasoning_effort)
response_base = # TODO

print("=== SENZA REASONING ===")
# TODO: stampa la risposta
# TODO: stampa i token totali

print()

# --- Chiamata CON reasoning ---
# TODO: fai la chiamata con reasoning_effort="high"
response_reasoning = # TODO

print("=== CON REASONING ===")
# TODO: stampa il reasoning (message.reasoning) se disponibile
# TODO: stampa la risposta
# TODO: stampa i token totali

print()

# --- Confronto ---
# TODO: calcola e stampa la differenza di token tra le due chiamate


---
## Fatto?

Se hai completato tutti gli esercizi, confronta le tue soluzioni con il notebook **09c** (soluzioni).

Nel prossimo blocco (Notebook 09d) vedremo come dare **superpoteri** all'LLM con il **function calling** e il protocollo **MCP**.
